In [1]:
%run Latex_macros.ipynb

<IPython.core.display.Latex object>

# Reasoning traces/Chain of Thought (CoT)

Consider a task 
- described by prompt $\x$ (a sequence)
- with response $\y$ (a sequence)

The most direct solution of the task can be described by the following sequence
describing the LLM's computation

$$
\x,  \y
$$

If the task is sufficiently complicated
- e.g., is best solved by multi-step reasoning

it has been shown that the chances of generating a better $\hat\y$ (sequence) answer
are improved by "Chain of Thought" reasoning.

**CoT summary**

Given prompt
$$
\x_{(1:\bar T)}
$$
rather than *immediately* producing response
$$
\y_{(1:\bar T)}
$$
giving computation trace
$$
\x_{(1:\bar T)},  \y_{(1: T)}
$$

an LLM is trained to think "Step by Step"
- creating a sequence of steps
- the Chain of Thought ("reasoning" trace) $\rat$
- enumerating sequential steps of a process that produces the response
$$
    \rat =  [ \rat_{(1)}, \ldots, \rat_{(\text{num_thoughts})} ]
$$
- where each $\rat_\tp$ is a thought represented as multi-token sequence 

resulting in trace

$$
\x, \rat, \y
$$

(We drop the indices of the sequences for clarity)

Thus, the model computes response $\y$ as
$$
\pr{\y\, | \x ,  \rat }
$$
rather than directly as
$$
\pr{\y \, | \x }
$$

The reasoning trace $\rat$
- *conditions* $\y$
- on the reasoning steps
- improving Performance




More formally, the LLM's "thought process" can be represented as a 
concatenation 
$$
\x, \rat, \y
$$
of
- prompt $\x$
- the *reasoning trace* $\rat$
    - a sequence of *thoughts* ordered linearly: the Chain of Thought
    
    $$
    \rat =  [ \rat_{(1)}, \ldots, \rat_{(\text{num_thoughts})} ]
    $$
    

- response $\y$


Note that each thought $\rat_\tp$ is a sequence of tokens.

    

# A model that reasons: "Think before speaking"

Rather than immediately outputting the "final" response
- we give the model the ability to "think"
    - create reasoning traces
- These thoughts can produce
    - multiple "preliminary" responses
- before returning the single "final" response



The thinking process can process can involve different strategies

- sequential
- parallel
- search/tree



**Note**

Syntactically:

The thoughts of the sequential strategy are easiest to display
- just a sequence
- parallel or tree strategies involve structure which complicates the display

By convention, thoughts are bracketed by the "think" delimiter

$$
\begin{array} \\
\bthink \\
\text{Thinking step-by-step, the first step is }\ldots \\
\ethink
\end{array}
$$

This gives the option
- to suppress the reasoning trace when the user only wants to see the response

Here is an example.

The prompt is

    Prove to me that the residuals of Linear Regression are not correlated with the independent variables
    
Note
- the reasoning trace is truncated (ran out of credits on HF before I could write the entire file)
- output: highlight the first words of each paragraph
    - indicates
        - thinking step by step
        - revision, etc

In [2]:
import re

def format_file_content(filename):
    try:
        with open(filename, 'r') as file:
            content = file.read()
        
        paragraphs = content.split('\n\n')
        formatted_paragraphs = []
        
        for paragraph in paragraphs:
            if paragraph.strip() in ['<think>', '</think>']:
                formatted_paragraphs.append(paragraph)
            else:
                # Find the first comma or period
                match = re.search(r'[,.]', paragraph)
                if match:
                    index = match.start()
                    formatted_paragraph = f"**{paragraph[:index]}**{paragraph[index:]}"
                else:
                    formatted_paragraph = f"**{paragraph}**"
                formatted_paragraphs.append(formatted_paragraph)
        
        return '\n\n'.join(formatted_paragraphs)
    except FileNotFoundError:
        return f"Error: The file '{filename}' was not found."
    except IOError:
        return f"Error: There was an issue reading the file '{filename}'."

In [3]:
# Usage
prompt = "Prove to me that the residuals of Linear Regression are not correlated with the independent variables"
print(prompt)
print("\n")
r1_out_file = 'r1_prove_residuals_orthogonal.txt'

formatted_content = format_file_content(r1_out_file)

# formatted_content = format_file_content('output.txt')
print(formatted_content)

Prove to me that the residuals of Linear Regression are not correlated with the independent variables


<think>

**Okay**, so I need to prove that the residuals from a linear regression aren't correlated with the independent variables. Hmm, let me start by recalling what residuals are. They're the differences between the actual observed values of the dependent variable and the values predicted by the model. Right, so residuals = y - ŷ.

**Now**, in linear regression, we have the model y = Xβ + ε, where X is the matrix of independent variables, β is the coefficient vector, and ε is the error term. The residuals are estimates of these errors, right? But wait, the errors are assumed to be uncorrelated with the independent variables in the classical linear regression model. Is that part of the Gauss-Markov assumptions? Yeah, I think one of the assumptions is that the error term has zero conditional mean, meaning E(ε|X) = 0. That implies that the errors are uncorrelated with the independent

# Training an LLM to reason


In order to produce an LLM that "reasons" given prompt $\x$
- we train it to produce responses
$$\rat, \y$$
- rather than the direct responses
$$\y$$
by training it on examples of the form

$$
\x, \rat, \y
$$
- rather than
$$
\x,  \y
$$


The cost of producing examples with reasoning traces 
$$
\x, \rat, \y
$$
can be substantial.

Often, it involves asking a human to adapt a training set with examples
$$
\x, \y
$$
by creating the reasoning traces $\rat$

## Reducing the cost of creating examples with reasoning traces

Removing the human from the training data production process is
highly desirable (cost reduction).

Boot-strapping a training dataset from an existing model is one solution.

One method of boot-strapping:

Ask a strong, non-reasoning LLM $\model^\text{non-reasoner}$ to solve task
- given $\x^\ip$
- "think step by step" (create $\rat^\ip$)
- produce response $\y^\ip$

**Note**
that Chain of Thought ("think step by step") reasoning
- seems to emerge from the natural training examples used to train $\model^\text{non-reasoner}$ 
- does *not* require explicit training examples with reasoning traces
- hence, the boot-strapping process is well grounded
    - non-circular: the first model is trained without reasoning traces

Alternatively:

Ask an existing reasoning model $\model^\text{reasoner}$ to solve task
- given $\x^\ip$
- produce response $\rat^\ip, \y^\ip$

This effectively bootstraps from existing reasoner $\model^\text{reasoner}$
- base case: *someone* needs to (manually) create the dataset to train the first reasoner

Both these approaches creates a machine-generated example
    $$\langle \x^\ip, \rat^\ip, \y^\ip \rangle$$
that can be used to train a new, reasoning LLM

## Distillation

The latter approach describes a use of the process called *Distillation*

We start by
- training model $\model^\text{teacher}$ to solve a task
- given training dataset $\langle \X, \y \rangle$ 
    - examples of how the task maps input $\x^\ip$ to response $\y^\ip$
for $1 \le i \le m$

We can then train a model $\model^\text{student}$
- to *mimic* $\model^\text{teacher}$
- using $\model^\text{teacher}$
    - on new inputs $\dot\X^\ip$
    - to produce outputs $\dot\y^\ip$
    - creating dataset $\langle \dot\X, \dot\y \rangle$
- and training $\model^\text{student}$ on $\langle \dot\X, \dot\y \rangle$

Note that 
- $\model^\text{student}$ does not directly learn the task
- it only learns to mimic $\model^\text{teacher}$
- inheriting all the flaws and limitations of the teacher


 $\model^\text{student }$ leverages the hard work used to train $\model^\text{teacher}$.
 
 Often
- $\model^\text{teacher}$ is an existing model
    - trained at great expense by someone else
- $\model^\text{teacher}$ is much larger (in number of parameters) than $\model^\text{student}$

# Reducing the cost of running a reasoning model

This is another use of Distillation
- Distill the knowledge of a large (high number of parameters) model $\model^\text{teacher}$
- Into a smaller model $\model^\text{student}$

Reasoning models
- with relatively small number (tens of billions) parameters
- have been distilled from much larger reasoning models (hundreds of billions parameters)
- with small decreases in performance

# Thinking harder

The reasoning trace $\rat$ is a sequences of thoughts.

How long should a model "think" ?
- what is the length (measured in tokens of  the reasoning traces $\rat$ ?

The initial approaches to "reasoning" favored
- long thoughts (thinking "harder" or "deeper")

So one approach to a better Reasoner is getting it to produce longer reasoning traces


## Thinking budget

**Reference**

[s1: Simple test-time scaling](https://arxiv.org/pdf/2501.19393)

An [interesting approach](https://arxiv.org/pdf/2501.19393) is to force a Reasoner
to obey a thinking budget for its reasoning traces
- measured in number of thoughts, or tokens
    - minimum
    - maximum
    


This approach would fall into the "search" class of using test-time compute.

Rather than training the Reasoner to obey a budget
- this approach **modifies the Inference loop**
    - the loop enforcing the auto-regressive behavior of the LLM
- to 
    - force the model to continue inference if the reasoning trace is below budget
    - truncate inference when the budget is exceeded

Forcing an overly long reasoning trace to adhere to maximum length is straight-forward
- Truncate the trace
- Insert an `<eos>` token

When the reasoning trace produced is too short
- the modified inference loop
    - replaces the token
    - with a token sequence that causes the model to revise/extend the reasoning trace
        - e.g., "Alternatively", "Wait", "But"
    - thus causing the LLM to continue "thinking"


Here is what the result looks like:

<br>

<table>
    <center><strong>Budget Forcing</strong></center>
    <tr>
        <img src="images/budget_forcing.png">
    </tr>
    
Attribution: https://arxiv.org/pdf/2501.19393#page=4
</table>

### Code for enforcing a thinking budget

Here is some simplified code
- derived from the paper's [Github](https://github.com/simplescaling/s1/tree/main?tab=readme-ov-file#vllm-with-budget-forcing)
- note some minor formating differences
    - does not explicitly use `<think>` and `</think>` to denote reasoning trace
    - instead, uses `<|im_start|>think`
        - denotes this part of the "conversation" is "thinking mode"
        
Note
- the multiple uses of the LLM `generate` method
- to generate the next part of the output
    - auto-regressive loop

<table>
    <center><strong>
        Code: Budget Forcing
        <br>
        creating the reasoning trace
        </strong></center>
    
    # Constants  
    ignore_str = "Wait"
    max_tokens_thinking_tmp = MAX_TOKENS_THINKING

    # Generate the start of the reasoning trace: Change the Assistant's role to **think**
    prompt += "<|im_start|>think"
    o = model.generate(
        prompt,
        sampling_params=sampling_params
    )
     
    # Increase length of reasoning trace until length is at least MAX_TOKENS_THINKING
    if max_tokens_thinking_tmp > 0:
        for i in range(NUM_IGNORE): # Num of times to skip stop token
            # Append the last extension to the reasoning trace
            prompt += o[0].outputs[0].text
            
            # Insert a "Wait"
            prompt += ignore_str
            
            # Generate the next extension of the trace
            o = model.generate(
                prompt,
                sampling_params=sampling_params
            )
            
            # Reduce the remaining number of thinking tokens to generate
            max_tokens_thinking_tmp -= len(o[0].outputs[0].token_ids)
            
            
            ...
            
          
            
</table>

<table>
    <center><strong>
        Code: Budget Forcing
        <br>
        creating the final answer
        </strong></center>
    
    ### Final answer ###
    # Append the last extension to the reasoning trace
    prompt += o[0].outputs[0].text # You can also append "Final Answer:" here like we do for some evaluations 
                                   # to prevent the model from just continuing to reason in its answer
                                   # when early exiting
    
    # Create the "answer", which follows the reasoning trace
    ...
    
    o = model.generate(
        prompt,
        sampling_params=sampling_params,
    )
    print("With budget forcing:") # You will see that after the "Wait" in the reasoning trace it fixes its answer
    print(prompt + o[0].outputs[0].text)

</table>

## Smarter not longer

At first glance
- longer reasoning trace should be preferred to a shorter trace
    - "deeper" reasoning

There is some empirical evidence that shows
- first preliminary response *can* often lead to correct response
- **but** the initial reasoning trace is often prematurely abandoned
    - the model "under-thinks" and tries something else if the first approach continues for too long


Why does a potentially successful initial reasoning trace get abandoned by the Reasoner ?

Perhaps it is in the training dataset
- hard problems in the training set have long but unsuccessful reasoning traces
- the model learns to abandon long traces

The problem is the inability to distinguish between
- long traces of hard problems that fail to be solved
- long traces that are needed for less-hard, solvable problems

### How to think smarter

One approach is
- train a model to estimate the difficulty of a give task
- have the Reasoner adapt its test-time compute based on the difficulty
    - more/longer thoughts for harder tasks

Alternatively
- use a *Process Reward Model*
    - train a Reward Model to estimate
        - whether each step in the thought is advancing toward a good response
    - continue a thought only if the estimated reward is high
    
The training dataset for a Process Reward Model
- is expensive
- human labeling of the steps and rewards
    - reward limited to categorical (Positive/Negative/Neutral) versus continuous values

In [4]:
print("Done")

Done
